# RSX-E09 — Failure-aware answer finalization vs full regeneration (GPU)

**96 new intervention trials across twelve calibration problems** that were ambiguous in NB29H but never definite-wrong in NB29H. Separate from E08. Four intervention policies × two budgets (64, 128 tokens), yielding paired comparisons of evaluability, correctness and inference cost.

Policies: direct one-line answer, recontextualize incomplete answer, continue original completion prefix, and fresh independent re-solve. All are **new interventions**, not reruns of NB29H/E02/E03. Contextual variants may expose incorrect draft reasoning; report them separately. This notebook is exploratory, not pristine confirmation.

In [ ]:
import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E08E09'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned source:',CODE_COMMIT)

In [ ]:
import json,time,random,hashlib,math,zipfile,gc,os,re,ast,base64,zlib
from pathlib import Path
from types import SimpleNamespace
from collections import Counter
import torch, numpy as np, pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available():raise RuntimeError('Colab GPU required (T4 or better).')
if torch.cuda.get_device_properties(0).total_memory<12*2**30:raise RuntimeError('Need >=12 GiB GPU.')
drive.mount('/content/drive')
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PROMPTS={'concise':'\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.',
'answer_first':'\n\nStart with FINAL_ANSWER: <number> on the first line, then optionally explain. For Countdown show the expression using each number exactly once.',
'structured':'\n\nUse exactly this format: ANSWER: <number>\nCHECK: <short verification>. For Countdown also show an expression using all input numbers exactly once.'}
def digest(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def atomic_json(path,obj):
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(tmp,path)
def verify_save(rec):
    dest=RECS/(hashlib.sha256(rec['trial_id'].encode()).hexdigest()+'.json')
    if dest.exists():
        prior=json.loads(dest.read_text());assert prior==rec,'Conflict: refusing overwrite '+rec['trial_id']
    else:atomic_json(dest,rec)
def checked_load(tid):
    dest=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not dest.exists():return None
    r=json.loads(dest.read_text());assert r['trial_id']==tid and r['protocol_hash']==PROTOCOL_HASH
    return r
def recorded():
    v=[json.loads(p.read_text()) for p in sorted(RECS.glob('*.json'))]
    assert len(v)==len({r['trial_id'] for r in v})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in v)
    return v
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def render(ids):return tokenizer.decode(list(ids),skip_special_tokens=True).strip()
def make_prompt(content):return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def snapshot_export(name):
    out=Path('/content')/name
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and p.suffix!='.tmp':z.write(p,p.relative_to(ROOT))
    print('Exported',out,'records:',len(recorded()))
    files.download(str(out))
    return out

STUDY='E09_FinalizationRouter_calibration_v1'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/E09_Finalization_V1');RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
DATA=json.loads(zlib.decompress(base64.b64decode("eNrtW+tu47gVfhXBf7oXNeD9YnRbTItd7ALt/NhLW6BZZCmSSoRxLNeSJ5PO5Dn6QH2xHpKyLTlOYnkn2ZndAHOJKPLwkDz8Pn7UyduJNY1vJtN/vZ0slnUx85eT6duJrWezqqnq+dnSt8vryRTlk1fV3E2mE7Os2otL31Z2kk/mq8vCL0PzH/N1+7Oymp/75WJZzVuoTwmyptRMUmQxF8hbVFoujVCe0VJJqY1UBpUOES6xslJwihlzwrLCECMnW8NV6H/ZvCF6as2sKpamBRenW4+mCGGUGlwuQuffX/ilzwz8FSSrnJ9DJTPLivqNb/LMG3uR2XremmoOPmdEZQs/b06yr+ur7NLMr+NjbN4mQ7O2Pvfh5z9lX85ddgUdZ1998/LFX89evPzuH19+O83+kKbkjyfgRrOYVcGLnrNQ2pol2JhMsaTiBOa19ZeLmWn9ZEpuoE29Wlof1qBYuVhPi3ziX5vZKhmAN2beXPnlWb2szisomFxWTQP+g20/gyKYq7PGg/8w2LPSVLPVEgyWZtZ4qPEGnLJVG1bJzM6St5PpfDWb5ZOu8ll7vYAWk9U89eQd9HVmLovqfFWvmrTufgnWbb1cetturDctlLa3i1fW+qbZPMNoKncGriyhNA4q9X+1rOfnt8phVs793MMEgiNt/QrWZDJlBEqX9WoBfkKo7fi3L1YvaxcG1U1eWS2bFkrrVbtYdaGStXUL0ZHmJKvLtPxVk8EC2lVYI5cV19nlatZWi9l1CBmIhV79GFehyrA4mlnAz/B+GiLxf/+FWDvJTuen8/VT9kUWAuLIaG9r2LFnF6a5gDaFZaREzEnnSqxZUXhmkEOmVAUSRcm9EVpLJzRX2uGitB4XVpaOOOYINjxErvfQu9ZEM3xXHDdtvQB4ME1Yp4mvw7zH1QHPAyJIxXMiNcoxlSSnGJ6IQDinkuVYcKVyzmmuGNYk1NtWw0xoGV8OilNrhWWwAI+hBOUE51BCmBQqPocHCo1yJvFdb4li6UnAfzmGVxjgC0IXNsRhs/7u5Z+J/vpdP5beafEuTRjMw5WZzWAPAraEqWAnmKqbm/yxINYAkDmBPTJMEKp8YVwhvOdYeEERL6n3VhgkiBXEIQmgSmSpSoJ0QaUkxdigY/shVumHIRbLJ4VYjukzxP72IBYiEUAVywSx66cAsRAQR0b7bxhioRbrg2iCzTXE7n3bh9j4L1bjIJaNg1h6oshjQixHmlFsYM2wNkgbQxGsdFkWTBcaGYttgbi02GnFNPHGy0JjVTpKuFNlUY4MOn4HxAp6wClWPCnECqqeIfZYiKVqC7ElVL/4oIH1JPshrFCKuK6Kf2NsO7vO6rn104C2p6fBknsLoXp62laXYJSImyPj/zcIuoCrGE5mcHANRbGWULB74CAXTDCwoHPoXvJcpwOuyjEGCA7cJjanXJxrMgpx+WjE5VI/IuIij5FSUlhTGm0Lhaw3pdYlD1cJhpSlMLDGnBp4JKIkMCNw0GUUeQmYS8feG8g77g3kAYdaip4ScQmV6BlxnxH3FuJCqK4Rl6KbI+P/GXEfQlwSzru7iAsYzMcirkRjEVfiEYBr69W8dfXVfIC3ROTgsrgTdnXJRIkkASy1YXkNxbSAwTuLMdLKYukAgTX2VjvDuKAeqZKCEBDwX8Hsw2G38Ssc82kfdX9ofG8vhHNDnsEfA4gJh9p+3MPey6CZW9nUImFjhik6yb717Wo5z0wW0SLbTnm2BYho06+R+Kf9UPzTAadfugPFYTDuspoD2j2D8sGgHGblwZuGfjw/hM511tSz1yk2unAM0Lpqhih8XIjl2RUYAFwLVWBvNnBAWEK0geOrph9v9cKniGliB7aGQc1TQbqZeFG2EdAXM1h+8MxVZQkrNG/7TWG+LzIDJpyrQkkOJUW7BE/jQ8cvcD5Jz656HQEhDWphgrkL34RzC7hdwiSC3xemjYPq7YjT009gBtYEIrLPYUJOTz/Nrurlq3C0Ab9+18RGBUD1q7iBwxjwSfa3juLCFAKdienG2HFo8Mtz0MzPz9uLHRoCyAcSwYJENsopoGBiGSFkLOGAkL0DOMabG2qoGO5JNq+2/EJxYCKdGIpgodfXJolS4C1XBP5TLFaFYy6wHeFAXUxCmYSfk21gMr25lBEoOEm1AKzH0EushiUMn9HYXgTLUMpQ+IEJxgKnAQ8qxNbuSqrl1gMJgc+Ijn5KBg5jKWHc66EO6BBKmFK98XIpuYLmUgUXuQCiBhZJY9ZadTQbRg3dYsT70xjpOxmNgxj2+hDjDuNrHOHKE8zVz2bccKYId2l3Uq7w3IcY194TpJgpVKG5LgmyZeFKbgqrldaghryCf6ml3EgDvKwpIkJRM26TEXUf5YKzGXibAJEfAojkqTmXPHPux825R8XYgHRh6iHI5nfEWWS3RLY+Mlaz7bsKQXplrlMlaAHUFUtjGO/j5DWv99k7jOtAOg6D3FLyQwyM5YaBMQ8UzEdS8HFY8PETbvokgfvfHIacix7mXHkH52ImZc6AvzrKDeQXmJASFt/CY3SEAnUSESQpeBArBCKLpQPztzk6Enpi8DC+u/k5jGjD0ffScjchQ5UaJ6bPzGg8M48hXKLGEi4Zdae4n3BFmuU7+dY7UVrDS+oKQFWliAMR64wRUjACM4a8kCVFyDlsiVOF8VC9YKgshLRS4nF7jIn7+FZsoRDTQ6BQyqdlW+jvmWzfK9nu3Do+CsUeEVa/WlULRLrhVJr9PhutasFAmsjpxtxxEPDR0+xW1N7m2KhYH+LYcHlKPgJR241ulz1hhExtVP2RojbaHk7bVtduOh5Ds0yMpllFf76u1dF7eSfPQpxSR01BsZOiKEKOmneUIO5RyR1nSGAiMS6Q0ZQ6gZQxppTIwdraslQjdS3n9+paDYjYXfTBIfsQzaGemGlDh89U+9FR7VGR9ehqVskPQc1uP0jicKGM5aOq2TUC/CrULOszLd5hWkIOUbOx/uOp2c78U6lZtk/N9u6Z46w8qprlfDTNkjFqNm62pvWLAxNktMUgSQUTslBUSauUhBimGMhVEgSPQLu8NJJTXkhmYLW91lZhKZQXJXl4c20cgsGjQX7MC0Cdpb8AJIHd64FsmsR+HMPZ3L4y5x52+DcBkqyvXsP+55k1y7Ze14OD+LpezG3IAw7Ms+aiWjQZ7dnYZtSsqy+BwKr5+8hc3M0NR88s+35vj/sB3fEteLiycKYCRN3JptkGVAMr1B4eUP3fLrgdViFHJ7wCOhrowC8yEe9Wt9ZfAAUmbm8icwdmArMtoEOX6nMeew6WMLCZ4CGhHAc5CY0jz4b4XcSMsI3dGNld0IY3XRJPFe1AaxCkNNqh0U4gwM8++zusbtmR7vSzz7JQ3PW6HcVx+/cDJceQ4kMRIHyugE5yxhKqdwqJYZXSdFBIKwcq0+u3ElQE0I4K7RTpfztNbWJGT+BHTNZkcpeyW6esh++M63vRTc8UUYkSO0bFGghFAa/FtCQhtOgZFxtSSqZ6ufCJs4AYuQCaEjQQY8hFWucQpe6SfaBS1uUshTynNRGuaW/TaGgeCtbmAy1TynJBKNXRSWBsGAjMLgmZUhtXaTfTg1l5iBmHwdUx43Z738uLoPwejxdLgS1CgmtUuBKOHBJOMSGFqbTawxFMMedLR5llXAhjSwWHEUwLLalyFJVejdxXesiLjb2o61lWrK7jJ5wAA74dghQLKX62miWMSsIB7a05qDhML43FcNKPyPQ+skrJzldV/MyGvyAb7sktTSt+eHrpOqD2JZhGWyHHNFWKMZhOYKvLy7WpTk8mUurFMpjYieHp7j0rA0ojGAGlQUs0aNk17LVD23aBCVOr7+MEdJ2khCKo+HnYKWCapErb3NfjNu0HTIbDfFeAbqoeznclElF+O981tY5UiADfO90VyEmA1uJrQosCb63n0k1kMphyZhPD6M5aZNY995Y7tKq39JTYiwd6ogx1D7c6YALjPdbjZ9lb8q/HfBu7kuv1iPvpQqlOTy2GLnuuyc5EkoybdN9RLKjHsSAR9BHVIbCcsrpkSCFvXekdF0YooUvEBQKxzRWmEPHSCF8gXXATfmvNMRDFltASjdtQmBykDtFh6pCQe9QhHJmfRh4y9iwPn5oQ7/t1izsEIhojEG9H1tVFBWewSLfNUCRC3aDL0OEqEXzZSsRIVpGt4IS/Y2MoFnsRfb9aFAjUItQGk4wFudhnQCg6bs9+fIqQj1WECd+HihBwnlHB+N2qMKWZDtUVP1gYbnzdUYdDEhryULrqvdXHrlBM1Q7QisnaRivGgv6AdPgTeJOHW9/eL7msX46hP0zGf4V8xOtRh6UXSmHOPadEOIuYBrUnnMEwUdyxAmFFmGMce224QwKeLCIl8RRT5kZuJnqPDCR7xR29LQPF3oroKWUgFs8y8ENivfcuBKtlEHYLmAqgqrUYi585mgNk4FfBx3zbdfo+2vlUQvzHgl7EB8lHO7GXrjHh5eAeFnOx5rKX/s3DxsXANhrY3v4CS2TeSLzRMtB2+gR7VWf/XoUwbKvwObX7tdzpcbv916sflSAA/zx+rYuCjPGgEA8Qjjyqw/DFEjHZkVMo4qj7Dtdxj9wjLrfCr3dPmSrfe2XbiVW85TLF5ENOhGHc8oDv8+D2L7gMebsj7a5vmJ/NtIa7VSEFzLwSa6OjOJWO/+RIbm5+fJzo7Ghg8q1fmGr5na0X/p9n0b+zv1x4+2pRA/Vmn+BPT/5TLSY3/we190YN")))
CASES=DATA['cases']; CASE_BY_ID={c['problem']['problem_id']:c for c in CASES}
assert len(CASES)==12 and all(c['problem']['split']=='calibration' and c['source']['evaluation']['failure_type']=='unanswered_or_ambiguous' for c in CASES)
ACTIONS=['direct_one_line','contextual_finalizer','prefix_continuation','fresh_resolve']
BUDGETS=[64,128]
PLAN=[{'problem_id':pid,'action':action,'budget':budget} for pid in sorted(CASE_BY_ID) for budget in BUDGETS for action in ACTIONS]
assert len(PLAN)==96
CONDITION={'study':STUDY,'parent_h_protocol':DATA['protocol_hash'],'code_commit':CODE_COMMIT,'model':MODEL_ID,'model_rev':MODEL_REV,
'policies':ACTIONS,'budgets':BUDGETS,'plan':PLAN,'q':Q,'tau':TAU,'window':WINDOW,'seed':62091}
PROTOCOL_HASH=digest(CONDITION)
if (ROOT/'protocol.json').exists():assert json.loads((ROOT/'protocol.json').read_text())=={'hash':PROTOCOL_HASH,'condition':CONDITION}
else:atomic_json(ROOT/'protocol.json',{'hash':PROTOCOL_HASH,'condition':CONDITION})
if (ROOT/'frozen_plan.json').exists():assert json.loads((ROOT/'frozen_plan.json').read_text())==PLAN
else:atomic_json(ROOT/'frozen_plan.json',PLAN)
print('Ready:',len(PLAN),'saved:',len(recorded()))

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Pinned ALoDLM loaded in FP32')

In [ ]:
# The model gets the original problem and (only for contextual actions) a draft output.
# Do not display the gold numeric answer or use gold correctness to pick actions.
def intervention(case,action,budget):
 p=case['problem'];r=case['source']; base=p['prompt']
 if action=='direct_one_line':
    prompt=base+'\n\nDo not provide working. Reply with one line containing the final numerical answer. For Countdown provide a valid expression before the final number.'
    return make_prompt(prompt),[],int(budget)
 if action=='fresh_resolve':
    prompt=base+'\n\nIndependently solve the problem from the beginning. Verify any computation. End with FINAL_ANSWER: <number>. For Countdown give a valid expression using each number once.'
    return make_prompt(prompt),[],int(budget)
 if action=='contextual_finalizer':
    # Cap text to avoid unbounded contexts, preserve the beginning+end for diagnostic consistency.
    prior=str(r['output']);prior=(prior[:380]+'\n[...]\n'+prior[-380:]) if len(prior)>800 else prior
    prompt=base+'\n\nAn incomplete draft is included for reference; verify it independently.\nDRAFT:\n'+prior+'\nEND DRAFT\nReturn a short checked final answer; for Countdown give a valid expression.'
    return make_prompt(prompt),[],int(budget)
 if action=='prefix_continuation':
    # Continue exact original completion with extra tokens, not a regenerated answer.
    prompt=base+PROMPTS[r['mode']]
    return make_prompt(prompt),list(map(int,r['token_ids'])),int(budget)
 raise ValueError(action)
assert len({c['problem']['problem_id'] for c in CASES})==12
print('Interventions initialized, no target leakage in instructions.')

In [ ]:
SESSION_CAP=8;SESSION_MINS=26
started=time.monotonic();n_new=0;errors=[]
for item in PLAN:
 if n_new>=SESSION_CAP or time.monotonic()-started>=SESSION_MINS*60:break
 pid=item['problem_id'];action=item['action'];budget=item['budget'];tid=f'E09|{pid}|{action}|{budget}|62091'
 if checked_load(tid) is not None:continue
 case=CASE_BY_ID[pid];p=case['problem'];src=case['source']
 try:
  tokens,prefix,max_new=intervention(case,action,budget)
  seed_all(62091);cfg=DecodeConfig(q=Q,tau=TAU,mode='left1',window_size=WINDOW,max_new_tokens=max_new,position_penalty=None)
  cfg.validate();torch.cuda.reset_peak_memory_stats();t0=time.monotonic()
  out=decoder.generate(tokens+prefix,cfg)
  new_ids=list(map(int,out['token_ids']));response=render(prefix+new_ids)
  ev=grade_output(response,p)
  rec={'trial_id':tid,'protocol_hash':PROTOCOL_HASH,'problem_id':pid,'split':'calibration','kind':p['kind'],
   'source_trial_id':src['trial_id'],'action':action,'budget':budget,'context_tokens':len(tokens),
   'preserved_tokens':len(prefix),'actual_generated_tokens':int(out.get('generated_tokens',len(new_ids))),
   'loop_token_passes':int(out.get('loop_token_passes',0)),'outer_steps':int(out.get('outer_steps',0)),
   'stop_reason':str(out.get('stop_reason','unknown')),'wall_seconds':round(time.monotonic()-t0,3),
   'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30),
   'evaluation':ev,'output':response,'seed':62091,'preservation_invariant':(prefix+new_ids)[:len(prefix)]==prefix}
  assert rec['preservation_invariant'];verify_save(rec);n_new+=1
  print('NEW',n_new,action,budget,pid,ev['failure_type'],flush=True)
 except torch.cuda.OutOfMemoryError:
  torch.cuda.empty_cache();errors.append({'trial_id':tid,'error':'OOM'});break
 except Exception as e:
  errors.append({'trial_id':tid,'error':repr(e)});break
if errors:atomic_json(ROOT/'latest_errors.json',errors)
print('New trials',n_new,'Total',len(recorded()),'/',len(PLAN))

In [ ]:
records=recorded();rows=[]
for r in records:
 rows.append({'problem_id':r['problem_id'],'action':r['action'],'budget':r['budget'],
 'correct':int(r['evaluation']['success']),'wrong':int(r['evaluation']['eligible_semantic_failure']),
 'ambiguous':int(r['evaluation']['failure_type']=='unanswered_or_ambiguous'),
 'forward_passes':r['loop_token_passes'],'context_tokens':r['context_tokens'],
 'seconds':r['wall_seconds']})
if rows:
 df=pd.DataFrame(rows);display(df.groupby(['budget','action']).agg(trials=('correct','size'),correct=('correct','sum'),
  wrong=('wrong','sum'),ambiguous=('ambiguous','sum'),mean_passes=('forward_passes','mean')).reset_index())
 df.to_csv(ROOT/'raw_results.csv',index=False)
 df.groupby(['budget','action']).agg(trials=('correct','size'),correct=('correct','sum'),wrong=('wrong','sum'),
  ambiguous=('ambiguous','sum'),mean_passes=('forward_passes','mean')).reset_index().to_csv(ROOT/'policy_summary.csv',index=False)
atomic_json(ROOT/'progress.json',{'planned':len(PLAN),'completed':len(records),'exploratory':True,
'unique_calibration_problems':len(CASES),'missing':[f"E09|{x['problem_id']}|{x['action']}|{x['budget']}|62091" for x in PLAN if checked_load(f"E09|{x['problem_id']}|{x['action']}|{x['budget']}|62091") is None]})
snapshot_export('RSX_E09_Finalization_Checkpoint.zip')